# QLoRA fine-tune Qwen2.5-3B-Instruct with Unsloth

QLoRA fine-tune `Qwen2.5-3B-Instruct` on the router dataset using Unsloth.
Run on **Kaggle** (free T4 x2) or **Google Colab** (free T4).

## Setup
1. New notebook, enable GPU accelerator (Kaggle: T4 x2, Colab: T4).
2. Run the install cell below (`!pip install unsloth`).
3. Make `HF_TOKEN` available: add a Kaggle Secret named `HF_TOKEN`, or on Colab run the auth cell and paste your token interactively.
4. In the config cell set `HF_DATASET` (source), `MODEL_REPO` (LoRA adapter output), and `GGUF_REPO` (GGUF output) to your repo ids.
5. Run all cells top to bottom.

This produces three artifacts: a trained LoRA adapter pushed to `MODEL_REPO`, a `q4_k_m` GGUF pushed to `GGUF_REPO`, and instructions (final section) to register the GGUF with Ollama **on your local machine**.

Each step lives in its own cell so the pipeline is reproducible and each stage can be run and inspected independently.

## 1. Install dependencies

Separated so it can be skipped on re-runs once the environment is set up.

In [ ]:
!pip install unsloth

## 2. Config

All tunable parameters in one place.

In [ ]:
HF_DATASET = "kostascherv/cortex-router-dataset"  # private repo; HF_TOKEN must have read access
BASE_MODEL = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
OUTPUT_DIR = "./router-lora"
GGUF_DIR = "./router-gguf"
MAX_SEQ_LEN = 512
LORA_RANK = 16
LORA_ALPHA = 32
BATCH_SIZE = 4
GRAD_ACCUM = 4
EPOCHS = 4
LR = 2e-4
MODEL_REPO = "kostascherv/cortex-router-model"       # LoRA adapter output repo
GGUF_REPO = "kostascherv/cortex-router-model-GGUF"   # GGUF output repo (for Ollama)
GGUF_FILENAME = "router-model-q4_k_m.gguf"           # must match FROM line in scripts/finetune/Modelfile

## 3. Authentication

Logs in to the HuggingFace Hub. Uses `HF_TOKEN` from the environment when present (Kaggle Secret / Colab env var); otherwise falls back to interactive `notebook_login()` (paste your token when prompted on Colab). The token must have **write** access since we push the adapter and GGUF.

In [ ]:
import os
from huggingface_hub import login, notebook_login

hf_token = os.environ.get("HF_TOKEN", "")
if hf_token:
    login(token=hf_token)
else:
    # No HF_TOKEN in env (e.g. plain Colab) -> interactive login.
    notebook_login()

## 4. Load base model

Load the base model in 4-bit and apply the Qwen 2.5 chat template.

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

## 5. Attach LoRA adapter

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_ALPHA,
    lora_dropout=0.0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 6. Load and format dataset

Load the dataset and render each example's `messages` into a single `text` field via the chat template.

In [ ]:
dataset = load_dataset(HF_DATASET)
train_dataset = dataset["train"]


def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}


train_dataset = train_dataset.map(apply_chat_template, remove_columns=["messages"])

## 7. Train

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        warmup_steps=10,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        fp16=True,
        logging_steps=10,
        output_dir=OUTPUT_DIR,
        save_strategy="epoch",
        report_to="none",
    ),
)
trainer.train()

## 8. Save adapter and push to the Hub

Save the LoRA adapter locally, then push it to `MODEL_REPO` (private) so it can be reloaded later without retraining.

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Push the LoRA adapter to the Hub (private by default).
model.push_to_hub(MODEL_REPO, private=True)
tokenizer.push_to_hub(MODEL_REPO, private=True)
print(f"Adapter saved to {OUTPUT_DIR} and pushed to {MODEL_REPO}")

## 9. Export GGUF and push to the Hub

Export a `q4_k_m`-quantized GGUF, then upload it to `GGUF_REPO` under the canonical filename `GGUF_FILENAME` (which matches the `FROM` line in `scripts/finetune/Modelfile`). Unsloth names the output file from the base model, so we discover the produced `.gguf` instead of hardcoding it.

In [ ]:
import glob

from huggingface_hub import HfApi

model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

# Unsloth names the GGUF after the base model, so locate it rather than hardcoding.
gguf_candidates = glob.glob("**/*.gguf", recursive=True)
if not gguf_candidates:
    raise FileNotFoundError("No .gguf file produced by save_pretrained_gguf")
gguf_path = max(gguf_candidates, key=os.path.getmtime)
print(f"GGUF produced at: {gguf_path}")

# Upload under the canonical name expected by scripts/finetune/Modelfile.
api = HfApi()
api.create_repo(GGUF_REPO, repo_type="model", private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=gguf_path,
    path_in_repo=GGUF_FILENAME,
    repo_id=GGUF_REPO,
    repo_type="model",
)
print(f"Uploaded {GGUF_FILENAME} to {GGUF_REPO}")

## 10. Register the model with Ollama (run locally)

The GPU notebook (Kaggle/Colab) does **not** have Ollama. Run the commands below on your **local machine**, from the repo root. They download the GGUF from the Hub into `scripts/finetune/` (next to `Modelfile`, whose `FROM ./router-model-q4_k_m.gguf` matches `GGUF_FILENAME`) and register it with Ollama:

```bash
huggingface-cli download kostascherv/cortex-router-model-GGUF router-model-q4_k_m.gguf --local-dir scripts/finetune
ollama create cortex-router -f scripts/finetune/Modelfile
ollama run cortex-router
```

Then point the app's router at it (`ROUTER_OLLAMA_MODEL=cortex-router`) and verify with `python -m scripts.score_router`.